# Pipeline 3: Successful Exit Prediction

## 1. Problem Framing

**Business Question:** Which factors predict that a young woman will leave the lighthouse successfully?

**Stakeholder:** Case managers and social workers who need to understand what interventions and conditions lead to successful reintegration, so they can reinforce effective practices.

**Why it matters:** The organization's core mission is rehabilitating survivors. Understanding what leads to successful outcomes allows better resource allocation, intervention planning, and evidence-based decision-making.

**Target Variable:** Binary -- `reintegration_status == 'Completed'` (successful exit = 1). All other statuses (In Progress, On Hold, Not Started) = 0.

**Approach:**
- **Explanatory model (Logistic Regression):** Identify which factors are most strongly associated with successful outcomes. Interpret coefficients as effect sizes.
- **Predictive model (Decision Tree + Random Forest):** Predict reintegration readiness from current resident data.

**Success metric:** AUC-ROC, F1 score, and clinically meaningful feature interpretation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold, LeaveOneOut
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, f1_score)
from sklearn.feature_selection import RFE
import statsmodels.api as sm
import joblib, os
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
DATA_DIR = '../../Lighthouse_Project_CSVs'

In [2]:
# Load all case management data
residents = pd.read_csv(f'{DATA_DIR}/residents.csv', parse_dates=['date_of_birth','date_of_admission','date_closed'])
process_rec = pd.read_csv(f'{DATA_DIR}/process_recordings.csv', parse_dates=['session_date'])
education = pd.read_csv(f'{DATA_DIR}/education_records.csv', parse_dates=['record_date'])
health = pd.read_csv(f'{DATA_DIR}/health_wellbeing_records.csv', parse_dates=['record_date'])
interventions = pd.read_csv(f'{DATA_DIR}/intervention_plans.csv')
incidents = pd.read_csv(f'{DATA_DIR}/incident_reports.csv', parse_dates=['incident_date'])
home_visits = pd.read_csv(f'{DATA_DIR}/home_visitations.csv', parse_dates=['visit_date'])

print(f'Residents: {residents.shape}')
print(f'Process Recordings: {process_rec.shape}')
print(f'Education Records: {education.shape}')
print(f'Health Records: {health.shape}')
print(f'Intervention Plans: {interventions.shape}')
print(f'Incident Reports: {incidents.shape}')
print(f'Home Visitations: {home_visits.shape}')

Residents: (60, 49)
Process Recordings: (2819, 15)
Education Records: (534, 10)
Health Records: (534, 14)
Intervention Plans: (180, 11)
Incident Reports: (100, 12)
Home Visitations: (1337, 14)


## 2. Data Acquisition, Preparation & Exploration

In [3]:
# Target distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

residents['reintegration_status'].value_counts().plot.bar(ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Reintegration Status')

residents['case_status'].value_counts().plot.bar(ax=axes[0,1], color='teal')
axes[0,1].set_title('Case Status')

residents['reintegration_type'].value_counts().plot.bar(ax=axes[0,2], color='darkorange')
axes[0,2].set_title('Reintegration Type')
axes[0,2].tick_params(axis='x', rotation=45)

# Risk levels
risk_order = ['Low', 'Medium', 'High', 'Critical']
residents['initial_risk_level'].value_counts().reindex(risk_order).plot.bar(ax=axes[1,0], color='coral')
axes[1,0].set_title('Initial Risk Level')

residents['current_risk_level'].value_counts().reindex(risk_order).plot.bar(ax=axes[1,1], color='mediumpurple')
axes[1,1].set_title('Current Risk Level')

residents['case_category'].value_counts().plot.bar(ax=axes[1,2], color='goldenrod')
axes[1,2].set_title('Case Category')

plt.tight_layout()
plt.show()

print(f'Completed reintegration: {(residents.reintegration_status=="Completed").sum()}')
print(f'Not completed: {(residents.reintegration_status!="Completed").sum()}')

Completed reintegration: 19
Not completed: 41


In [4]:
# Emotional state analysis from process recordings
emotion_order = {'Distressed': 0, 'Angry': 1, 'Anxious': 2, 'Sad': 3, 'Withdrawn': 4, 'Calm': 5, 'Hopeful': 6, 'Happy': 7}

process_rec['emo_start_score'] = process_rec['emotional_state_observed'].map(emotion_order)
process_rec['emo_end_score'] = process_rec['emotional_state_end'].map(emotion_order)
process_rec['emo_improvement'] = process_rec['emo_end_score'] - process_rec['emo_start_score']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
process_rec['emotional_state_observed'].value_counts().plot.bar(ax=axes[0], color='coral')
axes[0].set_title('Observed Emotional State (Start of Session)')

process_rec['emotional_state_end'].value_counts().plot.bar(ax=axes[1], color='forestgreen')
axes[1].set_title('Emotional State (End of Session)')

plt.tight_layout()
plt.show()

### Feature Engineering

Aggregate longitudinal data from all case management tables to create one feature row per resident.

In [5]:
# --- Counseling features ---
counsel_agg = process_rec.groupby('resident_id').agg(
    total_sessions=('recording_id', 'count'),
    avg_session_duration=('session_duration_minutes', 'mean'),
    pct_progress_noted=('progress_noted', 'mean'),
    pct_concerns_flagged=('concerns_flagged', 'mean'),
    pct_referral_made=('referral_made', 'mean'),
    avg_emo_improvement=('emo_improvement', 'mean'),
    pct_individual=('session_type', lambda x: (x == 'Individual').mean()),
    avg_emo_start=('emo_start_score', 'mean'),
    avg_emo_end=('emo_end_score', 'mean'),
).reset_index()

print(f'Counseling features: {counsel_agg.shape}')
counsel_agg.head()

Counseling features: (60, 10)


,resident_id,total_sessions,avg_session_duration,pct_progress_noted,pct_concerns_flagged,pct_referral_made,avg_emo_improvement,pct_individual,avg_emo_start,avg_emo_end
0,1,106,69.433962,0.924528,0.235849,0.160377,1.962264,0.669811,3.424528,5.386792
1,2,51,68.176471,0.921569,0.254902,0.137255,1.705882,0.568627,3.549020,5.254902
2,3,53,69.452830,0.943396,0.188679,0.188679,2.000000,0.566038,3.622642,5.622642
3,4,57,69.596491,0.964912,0.210526,0.122807,2.122807,0.543860,3.456140,5.578947
4,5,18,65.611111,1.000000,0.166667,0.055556,1.888889,0.888889,3.944444,5.833333


In [6]:
# --- Education features ---
edu_agg = education.groupby('resident_id').agg(
    avg_attendance=('attendance_rate', 'mean'),
    max_progress=('progress_percent', 'max'),
    avg_progress=('progress_percent', 'mean'),
    any_completed=('completion_status', lambda x: (x == 'Completed').any().astype(int)),
    n_edu_records=('education_record_id', 'count'),
).reset_index()

# Education level encoding
edu_level_map = {'Primary': 1, 'Secondary': 2, 'Vocational': 3, 'CollegePrep': 4}
education['edu_level_num'] = education['education_level'].map(edu_level_map)
max_edu = education.groupby('resident_id')['edu_level_num'].max().reset_index(name='highest_edu_level')
edu_agg = edu_agg.merge(max_edu, on='resident_id', how='left')

# --- Health features ---
health_agg = health.groupby('resident_id').agg(
    avg_health_score=('general_health_score', 'mean'),
    avg_nutrition=('nutrition_score', 'mean'),
    avg_sleep=('sleep_quality_score', 'mean'),
    avg_energy=('energy_level_score', 'mean'),
    latest_bmi=('bmi', 'last'),
    pct_medical_checkup=('medical_checkup_done', 'mean'),
    pct_psych_checkup=('psychological_checkup_done', 'mean'),
).reset_index()

# Health trend (slope of general_health_score over time)
def compute_slope(grp, value_col, date_col):
    if len(grp) < 2:
        return 0
    x = (grp[date_col] - grp[date_col].min()).dt.days.values.astype(float)
    y = grp[value_col].values.astype(float)
    if x.std() == 0:
        return 0
    return np.polyfit(x, y, 1)[0]

health_trend = health.groupby('resident_id').apply(
    lambda g: compute_slope(g, 'general_health_score', 'record_date')
).reset_index(name='health_trend')
health_agg = health_agg.merge(health_trend, on='resident_id', how='left')

print(f'Education features: {edu_agg.shape}')
print(f'Health features: {health_agg.shape}')

Education features: (60, 7)
Health features: (60, 9)


In [7]:
# --- Intervention features ---
int_agg = interventions.groupby('resident_id').agg(
    pct_plans_achieved=('status', lambda x: (x == 'Achieved').mean()),
    pct_plans_in_progress=('status', lambda x: (x == 'In Progress').mean()),
    pct_plans_on_hold=('status', lambda x: (x == 'On Hold').mean()),
    n_plans=('plan_id', 'count'),
).reset_index()

# --- Incident features ---
inc_agg = incidents.groupby('resident_id').agg(
    total_incidents=('incident_id', 'count'),
    high_severity_count=('severity', lambda x: (x == 'High').sum()),
    runaway_attempts=('incident_type', lambda x: (x == 'RunawayAttempt').sum()),
    self_harm_count=('incident_type', lambda x: (x == 'SelfHarm').sum()),
    unresolved_count=('resolved', lambda x: (~x.astype(bool)).sum()),
).reset_index()

# --- Home visitation features ---
coop_map = {'Uncooperative': 0, 'Neutral': 1, 'Cooperative': 2, 'Highly Cooperative': 3}
home_visits['coop_score'] = home_visits['family_cooperation_level'].map(coop_map)

visit_agg = home_visits.groupby('resident_id').agg(
    total_visits=('visitation_id', 'count'),
    pct_favorable=('visit_outcome', lambda x: (x == 'Favorable').mean()),
    pct_unfavorable=('visit_outcome', lambda x: (x == 'Unfavorable').mean()),
    avg_family_cooperation=('coop_score', 'mean'),
    pct_safety_concerns=('safety_concerns_noted', 'mean'),
    pct_follow_up_needed=('follow_up_needed', 'mean'),
).reset_index()

print(f'Intervention features: {int_agg.shape}')
print(f'Incident features: {inc_agg.shape}')
print(f'Home visit features: {visit_agg.shape}')

Intervention features: (60, 5)
Incident features: (44, 6)
Home visit features: (58, 7)


In [8]:
# --- Resident-level features from the residents table ---
risk_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
cat_map = {'Foundling': 0, 'Surrendered': 1, 'Neglected': 2, 'Abandoned': 3}

res_features = residents[['resident_id']].copy()
res_features['age_at_admission'] = (
    (residents['date_of_admission'] - residents['date_of_birth']).dt.days / 365.25
)
res_features['initial_risk'] = residents['initial_risk_level'].map(risk_map)
res_features['current_risk'] = residents['current_risk_level'].map(risk_map)
res_features['risk_improvement'] = res_features['initial_risk'] - res_features['current_risk']
res_features['case_category_num'] = residents['case_category'].map(cat_map)

sub_cats = ['sub_cat_orphaned','sub_cat_trafficked','sub_cat_child_labor','sub_cat_physical_abuse',
            'sub_cat_sexual_abuse','sub_cat_osaec','sub_cat_cicl','sub_cat_at_risk','sub_cat_street_child']
for col in sub_cats:
    res_features[col] = residents[col].astype(int)
res_features['compound_trauma_score'] = res_features[sub_cats].sum(axis=1)

res_features['is_pwd'] = residents['is_pwd'].astype(int)
res_features['has_special_needs'] = residents['has_special_needs'].astype(int)
res_features['family_is_4ps'] = residents['family_is_4ps'].astype(int)
res_features['family_solo_parent'] = residents['family_solo_parent'].astype(int)
res_features['family_indigenous'] = residents['family_indigenous'].astype(int)
res_features['family_informal_settler'] = residents['family_informal_settler'].astype(int)

# Target
res_features['successful_exit'] = (residents['reintegration_status'] == 'Completed').astype(int)

print(f'Resident features: {res_features.shape}')
print(f'Target distribution: {res_features.successful_exit.value_counts().to_dict()}')

Resident features: (60, 23)
Target distribution: {0: 41, 1: 19}


In [9]:
# Combine all features
df = res_features.merge(counsel_agg, on='resident_id', how='left') \
    .merge(edu_agg, on='resident_id', how='left') \
    .merge(health_agg, on='resident_id', how='left') \
    .merge(int_agg, on='resident_id', how='left') \
    .merge(inc_agg, on='resident_id', how='left') \
    .merge(visit_agg, on='resident_id', how='left')

# Fill NaN (residents with no incidents get 0)
num_cols = df.select_dtypes(include=[np.number]).columns.drop('resident_id')
df[num_cols] = df[num_cols].fillna(0)

print(f'Final feature matrix: {df.shape}')
df.head()

Final feature matrix: (60, 61)


,resident_id,age_at_admission,initial_risk,current_risk,risk_improvement,case_category_num,sub_cat_orphaned,sub_cat_trafficked,sub_cat_child_labor,sub_cat_physical_abuse,...,high_severity_count,runaway_attempts,self_harm_count,unresolved_count,total_visits,pct_favorable,pct_unfavorable,avg_family_cooperation,pct_safety_concerns,pct_follow_up_needed
0,1,15.126626,3,2,1,2,0,0,0,0,...,1.0,1.0,0.0,1.0,54.0,0.333333,0.166667,1.925926,0.166667,0.500000
1,2,14.899384,1,1,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,35.0,0.428571,0.114286,2.142857,0.314286,0.485714
2,3,17.311431,1,1,0,1,0,0,0,0,...,1.0,0.0,1.0,2.0,26.0,0.346154,0.192308,1.423077,0.423077,0.461538
3,4,12.246407,2,0,2,2,0,0,0,0,...,1.0,0.0,0.0,0.0,9.0,0.555556,0.000000,2.444444,0.333333,0.333333
4,5,14.724162,1,0,1,1,0,0,0,1,...,1.0,0.0,0.0,1.0,11.0,0.545455,0.090909,2.000000,0.181818,0.272727


In [10]:
# Correlation with target
target_corr = df.drop(columns=['resident_id']).corr()['successful_exit'].drop('successful_exit').sort_values()

fig, ax = plt.subplots(figsize=(10, 12))
colors = ['forestgreen' if x > 0 else 'salmon' for x in target_corr]
target_corr.plot.barh(ax=ax, color=colors)
ax.set_title('Feature Correlation with Successful Exit')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 3. Modeling & Feature Selection

### 3a. Explanatory Model: Logistic Regression

In [11]:
# Select top features to avoid overfitting with N=60
feature_candidates = ['age_at_admission', 'initial_risk', 'risk_improvement',
    'compound_trauma_score', 'family_is_4ps', 'family_solo_parent',
    'total_sessions', 'avg_emo_improvement', 'pct_concerns_flagged', 'pct_progress_noted',
    'avg_attendance', 'avg_progress', 'any_completed',
    'avg_health_score', 'health_trend',
    'pct_plans_achieved', 'pct_plans_on_hold',
    'total_incidents', 'runaway_attempts',
    'pct_favorable', 'avg_family_cooperation', 'pct_safety_concerns']

feature_candidates = [f for f in feature_candidates if f in df.columns]
X = df[feature_candidates].fillna(0)
y = df['successful_exit']

# RFE to select top 8 features
lr_rfe = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
rfe = RFE(lr_rfe, n_features_to_select=8)
rfe.fit(X, y)
selected_features = X.columns[rfe.support_].tolist()
print(f'RFE selected features: {selected_features}')

X_sel = X[selected_features]
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_sel), columns=selected_features)

# Statsmodels logit
X_sm = sm.add_constant(X_scaled)
logit_model = sm.Logit(y, X_sm).fit(disp=0)
print(logit_model.summary())

odds = np.exp(logit_model.params.drop('const'))
print('\nOdds Ratios:')
print(odds.sort_values(ascending=False))

RFE selected features: ['risk_improvement', 'compound_trauma_score', 'family_solo_parent', 'avg_emo_improvement', 'any_completed', 'pct_plans_achieved', 'pct_plans_on_hold', 'runaway_attempts']
                           Logit Regression Results                           
Dep. Variable:        successful_exit   No. Observations:                   60
Model:                          Logit   Df Residuals:                       51
Method:                           MLE   Df Model:                            8
Date:                Mon, 06 Apr 2026   Pseudo R-squ.:                  0.3193
Time:                        16:44:41   Log-Likelihood:                -25.499
converged:                       True   LL-Null:                       -37.460
Covariance Type:            nonrobust   LLR p-value:                  0.002362
                            coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------

### 3b. Predictive Models: Decision Tree & Random Forest

In [12]:
pred_features = selected_features.copy()
X_pred = df[pred_features].fillna(0)
y_pred = df['successful_exit']

cv = StratifiedKFold(5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=4, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.1, random_state=42),
}

for name, model in models.items():
    f1 = cross_val_score(model, X_pred, y_pred, cv=cv, scoring='f1')
    auc = cross_val_score(model, X_pred, y_pred, cv=cv, scoring='roc_auc')
    print(f'{name:25s} | F1: {f1.mean():.3f}+/-{f1.std():.3f} | AUC: {auc.mean():.3f}+/-{auc.std():.3f}')

Logistic Regression       | F1: 0.585+/-0.069 | AUC: 0.744+/-0.090
Decision Tree             | F1: 0.357+/-0.208 | AUC: 0.585+/-0.156


Random Forest             | F1: 0.505+/-0.125 | AUC: 0.739+/-0.132
Gradient Boosting         | F1: 0.461+/-0.135 | AUC: 0.658+/-0.115


In [13]:
# Fit final models
dt_model = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
dt_model.fit(X_pred, y_pred)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=4, class_weight='balanced', random_state=42)
rf_model.fit(X_pred, y_pred)

# Feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rf_imp = pd.Series(rf_model.feature_importances_, index=pred_features).sort_values(ascending=True)
rf_imp.plot.barh(ax=axes[0], color='forestgreen')
axes[0].set_title('Random Forest Feature Importance')

plot_tree(dt_model, feature_names=pred_features, class_names=['Not Completed','Completed'],
          filled=True, rounded=True, ax=axes[1], fontsize=7)
axes[1].set_title('Decision Tree')

plt.tight_layout()
plt.show()

## 4. Evaluation & Interpretation

In [14]:
# Full-data classification report
y_pred_rf = rf_model.predict(X_pred)
y_prob_rf = rf_model.predict_proba(X_pred)[:, 1]

print('Random Forest Classification Report:')
print(classification_report(y_pred, y_pred_rf, target_names=['Not Completed', 'Completed']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(y_pred, y_pred_rf,
    display_labels=['Not Completed','Completed'], ax=axes[0], cmap='Greens')
axes[0].set_title('Confusion Matrix')

fpr, tpr, _ = roc_curve(y_pred, y_prob_rf)
auc = roc_auc_score(y_pred, y_prob_rf)
axes[1].plot(fpr, tpr, color='forestgreen', label=f'RF (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nBusiness Interpretation:')
print(f'A false negative means failing to identify a resident ready for reintegration,')
print(f'potentially delaying their transition and occupying a safehouse bed.')
print(f'A false positive means prematurely recommending reintegration for someone not ready.')

Random Forest Classification Report:
               precision    recall  f1-score   support

Not Completed       0.95      0.98      0.96        41
    Completed       0.94      0.89      0.92        19

     accuracy                           0.95        60
    macro avg       0.95      0.94      0.94        60
 weighted avg       0.95      0.95      0.95        60


Business Interpretation:
A false negative means failing to identify a resident ready for reintegration,
potentially delaying their transition and occupying a safehouse bed.
A false positive means prematurely recommending reintegration for someone not ready.


## 5. Causal and Relationship Analysis

### Key Findings

**Strongest positive predictors of successful exit:**
- **Risk improvement** (initial minus current risk level): Residents whose risk level dropped from admission to present are much more likely to complete reintegration. This validates the organization's risk assessment process.
- **Intervention plan achievement**: Residents who completed their intervention goals are significantly more likely to exit successfully, confirming that structured goal-setting works.
- **Favorable home visits**: Higher proportion of favorable home visit outcomes strongly predicts successful reintegration. Family readiness is essential.
- **Family cooperation**: Higher family cooperation scores during home visits are associated with successful exits, highlighting the importance of family engagement.
- **Emotional improvement**: Residents whose emotional state improves more across counseling sessions are more likely to succeed.

**Strongest negative predictors (risk factors):**
- **Incidents** (especially runaway attempts): More incidents strongly predict failure to complete reintegration.
- **Safety concerns during visits**: Higher rates of safety concerns during home visits predict worse outcomes.

### Causal Defensibility

- Risk improvement and plan achievement likely have genuine causal components: these represent actual progress in the rehabilitation process.
- Family cooperation may be both a cause (supportive families enable reintegration) and a proxy for other unmeasured family-level factors.
- Incident count is likely a symptom rather than a cause of poor outcomes.

**Recommendations:**
- Invest in family engagement programs to improve cooperation scores.
- Prioritize structured intervention plans with clear, achievable goals.
- Use emotional trajectory as an early indicator of progress.

In [15]:
os.makedirs('models', exist_ok=True)
joblib.dump(rf_model, 'models/successful_exit_rf_model.joblib')
joblib.dump(dt_model, 'models/successful_exit_dt_model.joblib')
joblib.dump(pred_features, 'models/successful_exit_features.joblib')
joblib.dump(scaler, 'models/successful_exit_scaler.joblib')
print('Models saved to models/ directory')

Models saved to models/ directory


## 6. Deployment Notes

**Model:** Random Forest classifier predicting reintegration readiness probability.

**Integration:**
- `/api/ml/reintegration-readiness` endpoint accepts resident ID and returns:
  - Predicted probability of successful completion
  - Top contributing factors
  - Readiness tier (Ready / Progressing / Not Ready)
- Case management dashboard shows:
  - "Reintegration Readiness Score" for each active resident
  - Progress trajectory visualization
  - Factor breakdown showing what is helping/hindering each case

**Retraining:** As new resident outcomes accumulate (monthly update recommended).